In [ ]:
import pandas as pd
import re
import json
import urllib
import requests
from sqlalchemy import create_engine
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
from geopy.distance import geodesic
import numpy as np

# --- Load SQL Credentials ---
with open("credentials_resv_data_access_SQL_20250528.json", "r") as f:
    config = json.load(f)
sql_config = config["sql_server_config"]
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={sql_config['server']};"
    f"DATABASE={sql_config['database']};"
    f"UID={sql_config['username']};"
    f"PWD={sql_config['password']}"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# --- Load SQL Metadata ---
with open("entire_dvrt_data_set_excluding_resv_with_metadata_inactive_and_active_obsv_and_manual_20250529.sql", "r") as f:
    sql_query = f.read()
metadata_df = pd.read_sql(sql_query, engine)

# --- Preprocess station names for TF-IDF ---
def preprocess(name):
    name = name.upper()
    name = re.sub(r'\(.*?\)|[^A-Z0-9\s]', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

metadata_df["processed_name"] = metadata_df["MasterStationName"].apply(preprocess)

# --- TF-IDF Grouping ---
print("🔍 Grouping similar stations using TF-IDF...")
grouped_tfidf = []
group_id = 0
vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(metadata_df["processed_name"])
cosine_sim = cosine_similarity(tfidf_matrix)

visited = set()
for i in tqdm(range(len(metadata_df)), desc="TF-IDF Grouping", unit="group"):
    if i in visited:
        continue
    group = [i]
    for j in range(i + 1, len(metadata_df)):
        if cosine_sim[i, j] >= 0.7:
            group.append(j)
    if len(group) > 1:
        group_id += 1
        grouped_tfidf.append((f"PD{str(group_id).zfill(3)}", metadata_df.iloc[group], "TF-IDF"))
        visited.update(group)

# --- Download time series data for Pearson correlation ---
print("🌐 Downloading discharge data from API...")
station_ids = metadata_df["STATION_ID"].dropna().unique().astype(str)
data_dict = {}
today = datetime.today().strftime("%Y-%m-%d")

for station_id in tqdm(station_ids, desc="Downloading", unit="station"):
    url = f"https://www.waterrights.utah.gov/dvrtdb/daily-chart.asp?station_id={station_id}&end_date={today}&f=json"
    r = requests.get(url)
    if r.status_code == 200:
        json_data = r.json()
        if "data" in json_data:
            df = pd.DataFrame(json_data["data"], columns=["date", "value"])
            df["date"] = pd.to_datetime(df["date"])
            df["value"] = pd.to_numeric(df["value"], errors="coerce")
            df = df.drop_duplicates(subset="date")
            df = df.set_index("date").rename(columns={"value": station_id})
            data_dict[station_id] = df

# --- Pearson Grouping (correlation ≥ 0.95) ---
grouped_corr = []
if data_dict:
    print("📈 Grouping similar stations using Pearson correlation...")
    combined_df = pd.concat(data_dict.values(), axis=1)
    combined_df = combined_df.interpolate(limit_direction="both")
    corr_matrix = combined_df.corr()
    corr_pairs = corr_matrix.stack().reset_index()
    corr_pairs.columns = ["Station1", "Station2", "Correlation"]
    corr_pairs = corr_pairs[(corr_pairs["Station1"] != corr_pairs["Station2"]) & (corr_pairs["Correlation"] >= 0.95)]

    used = set()
    for _, row in tqdm(corr_pairs.iterrows(), total=len(corr_pairs), desc="Pearson Grouping", unit="pair"):
        s1, s2 = row["Station1"], row["Station2"]
        if s1 in used or s2 in used:
            continue
        members = set([s1, s2])
        for s3 in corr_pairs[(corr_pairs["Station1"] == s1) | (corr_pairs["Station2"] == s1) |
                             (corr_pairs["Station1"] == s2) | (corr_pairs["Station2"] == s2)][["Station1", "Station2"]].values.flatten():
            members.add(s3)
        used.update(members)
        group_id += 1
        group_df = metadata_df[metadata_df["STATION_ID"].isin(members)]
        if len(group_df) > 1:
            grouped_corr.append((f"PD{str(group_id).zfill(3)}", group_df, "Pearson"))

# --- Transpose + Geo Columns ---
def transpose_group_with_geo(group_id, group_df, grouping_method):
    row = {
        "SiteID (New)": group_id,
        "NewSiteName": group_df.iloc[0]["MasterStationName"].title(),
        "GroupingMethod": grouping_method
    }

    station_ids = group_df["STATION_ID"].dropna().astype(str).tolist()
    row["StationPage"] = f'https://waterrights.utah.gov/dvrtdb/daily-chart.asp?STATION_ID={",".join(station_ids)}'

    max_stations = len(group_df)
    lat_list = []
    lon_list = []

    field_order = [
        ("MasterStationName", "MasterStationName"),
        ("Station_ID (old)", "STATION_ID"),
        ("UNITS_DESC_ENTRY", "UNITS_DESC_ENTRY"),
        ("CollectionStationName", "CollectionStationName"),
        ("CollectionSystemName", "COLLECTION_SYSTEM"),
        ("SYSTEM_NAME", "SYSTEM_NAME"),
        ("LAT", "LAT"),
        ("LON", "LON"),
        ("SiteType", "SiteType")
    ]

    for field_label, col_name in field_order:
        for i in range(max_stations):
            value = group_df.iloc[i][col_name]
            row[f"{field_label} {i+1}"] = value
            if field_label == "LAT":
                lat_list.append(value)
            if field_label == "LON":
                lon_list.append(value)

    row["LAT/LON MATCH"] = "Yes" if len(set(round(float(lat), 6) for lat in lat_list)) == 1 and len(set(round(float(lon), 6) for lon in lon_list)) == 1 else "No"

    origin = (float(lat_list[0]), float(lon_list[0]))
    for i in range(1, len(lat_list)):
        try:
            target = (float(lat_list[i]), float(lon_list[i]))
            feet = geodesic(origin, target).feet
            row[f"Distance {i} (feet)"] = round(feet)
        except:
            row[f"Distance {i} (feet)"] = np.nan

    row["Check"] = ""
    row["Comment"] = ""
    return row

# --- Combine and Export ---
print("📦 Structuring final transposed output...")
final_rows_all = []
final_rows_tfidf = []
final_rows_corr = []

for group_id, df_group, method in tqdm(grouped_tfidf + grouped_corr, desc="Transposing Groups", unit="group"):
    row = transpose_group_with_geo(group_id, df_group, method)
    final_rows_all.append(row)
    if method == "TF-IDF":
        final_rows_tfidf.append(row)
    elif method == "Pearson":
        final_rows_corr.append(row)

# --- Save all outputs ---
today_str = datetime.today().strftime("%Y%m%d")
pd.DataFrame(final_rows_all).to_csv(f"2.0_NonReservoir_Station_Naming_Metadata_Match_ALL_{today_str}.csv", index=False)
pd.DataFrame(final_rows_tfidf).to_csv(f"2.0_NonReservoir_Station_Naming_Metadata_TFIDF_{today_str}.csv", index=False)
pd.DataFrame(final_rows_corr).to_csv(f"2.0_NonReservoir_Station_Naming_Metadata_Pearson_{today_str}.csv", index=False)

print("✅ CSVs saved!")